# Modelado Predictivo Avanzado: Ensemble Adaptativo (Prophet, SARIMA, XGBoost)

**Documento Técnico Nivel Maestría/Doctorado - Análisis Metodológico Aplicado**

Este cuaderno implementa una metodología de **Ensemble Adaptativo** para predecir el recaudo de Rentas Cedidas. Combina la fortaleza de tres modelos competitivos: Prophet (tendencias seculares y shocks), SARIMA (estabilidad lineal y autocorrelación) y XGBoost (relaciones no lineales complejas y variables exógenas).

## Objetivo del Modelado Predictivo
Se implementa una metodología de Ensemble Adaptativo para predecir la variable \ValorRecaudo\ a partir de \BaseRentasCedidasVF.xlsx\, aplicando la metodología CRISP-DM con tres modelos competitivos: Prophet, SARIMA y XGBoost.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error

# Modelos
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
import xgboost as xgb

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.1)
warnings.filterwarnings('ignore')

# 1. Preprocesamiento e Ingeniería de Características (Crítico)
file_path = r'C:\Users\efren\Music\ESTRUCTURA DATOS RENTAS\BaseRentasCedidasVF.xlsx'
df = pd.read_excel(file_path)
df['FechaRecaudo'] = pd.to_datetime(df['FechaRecaudo'])
df.set_index('FechaRecaudo', inplace=True)

# Resampling mensual ('MS')
y_t = df['ValorRecaudo'].resample('MS').sum()
df_model = y_t.to_frame(name='Recaudo_Neto')

# Generación de Variables (Feature Engineering)
df_model['lag_12'] = df_model['Recaudo_Neto'].shift(12) # 23% de importancia
df_model['rolling_mean_6'] = df_model['Recaudo_Neto'].rolling(window=6).mean() # Tendencia mediano plazo
df_model['rolling_std_3'] = df_model['Recaudo_Neto'].rolling(window=3).std() # Volatilidad reciente

# Variables dummy calendáricas (Picos de festividades y primas)
df_model['Mes'] = df_model.index.month
df_model['is_jun_dec'] = df_model['Mes'].isin([6, 12]).astype(int)

# Eliminar NaNs generados por lags y ventanas móviles
df_model.dropna(inplace=True)

print(f"Dataset preparado para modelado: {len(df_model)} meses.")
df_model.head()

## 2. Arquitectura de Validación (Train/Test Split)
Dividimos los datos cronológicamente: 80% para entrenamiento y 20% para prueba (validación out-of-sample).

In [ ]:
# Train/Test Split (80/20 cronológico)
train_size = int(len(df_model) * 0.8)
train, test = df_model.iloc[:train_size], df_model.iloc[train_size:]

print(f"Tamaño Entrenamiento: {len(train)} meses")
print(f"Tamaño Prueba: {len(test)} meses")

# Función auxiliar para calcular métricas
def calculate_metrics(y_true, y_pred, model_name):
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return {'Model': model_name, 'MAPE (%)': mape, 'RMSE': rmse, 'MAE': mae}

## 3. Ejecución de Modelos Base
Entrenamos los tres modelos competitivos definidos en la metodología.

In [ ]:
resultados_metricas = []
predicciones_test = pd.DataFrame(index=test.index)
predicciones_test['Real'] = test['Recaudo_Neto']

# --- MODELO A: PROPHET ---
# Preparar datos para Prophet (requiere columnas 'ds' y 'y')
df_prophet_train = train.reset_index()[['FechaRecaudo', 'Recaudo_Neto']].rename(columns={'FechaRecaudo': 'ds', 'Recaudo_Neto': 'y'})
df_prophet_test = test.reset_index()[['FechaRecaudo']].rename(columns={'FechaRecaudo': 'ds'})

model_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
model_prophet.fit(df_prophet_train)

forecast_prophet = model_prophet.predict(df_prophet_test)
predicciones_test['Prophet'] = forecast_prophet['yhat'].values
resultados_metricas.append(calculate_metrics(test['Recaudo_Neto'], predicciones_test['Prophet'], 'Prophet'))

# --- MODELO B: SARIMA ---
# Baseline SARIMAX(1,1,1)(1,1,1,12)
model_sarima = SARIMAX(train['Recaudo_Neto'], 
                       order=(1, 1, 1), 
                       seasonal_order=(1, 1, 1, 12),
                       enforce_stationarity=False, 
                       enforce_invertibility=False)
res_sarima = model_sarima.fit(disp=False)

predicciones_test['SARIMA'] = res_sarima.forecast(steps=len(test))
resultados_metricas.append(calculate_metrics(test['Recaudo_Neto'], predicciones_test['SARIMA'], 'SARIMA'))

# --- MODELO C: XGBOOST ---
features = ['lag_12', 'rolling_mean_6', 'rolling_std_3', 'is_jun_dec']
X_train, y_train = train[features], train['Recaudo_Neto']
X_test, y_test = test[features], test['Recaudo_Neto']

model_xgb = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
model_xgb.fit(X_train, y_train)

predicciones_test['XGBoost'] = model_xgb.predict(X_test)
resultados_metricas.append(calculate_metrics(test['Recaudo_Neto'], predicciones_test['XGBoost'], 'XGBoost'))

## 4. Arquitectura de Ensemble Adaptativo
Creamos el Ensemble asignando pesos según la tipología municipal (Tipo A - Consolidados): 30% SARIMA, 50% XGBoost, 20% Prophet.

In [ ]:
# Ensemble Adaptativo (Pesos: 30% SARIMA, 50% XGBoost, 20% Prophet)
predicciones_test['Ensemble'] = (predicciones_test['SARIMA'] * 0.30) + \
                                (predicciones_test['XGBoost'] * 0.50) + \
                                (predicciones_test['Prophet'] * 0.20)

resultados_metricas.append(calculate_metrics(test['Recaudo_Neto'], predicciones_test['Ensemble'], 'Ensemble Adaptativo'))

# Mostrar Tabla de Métricas
df_metricas = pd.DataFrame(resultados_metricas).set_index('Model')
print("--- RENDIMIENTO DE MODELOS (TEST SET) ---")
print(df_metricas.round(2))

mape_ensemble = df_metricas.loc['Ensemble Adaptativo', 'MAPE (%)']
print(f"\n¿El Ensemble cumple la meta de MAPE < 11%? {'SÍ' if mape_ensemble < 11 else 'NO'} ({mape_ensemble:.2f}%)")

## 5. Output Esperado: Visualización y Alerta de Eficiencia
Generamos la gráfica comparativa con Intervalos de Confianza (simulados al 95% basados en el RMSE del Ensemble) y disparamos la 'Alerta de Eficiencia' si el recaudo real cae por debajo del límite inferior.

In [ ]:
# Calcular Intervalos de Confianza (95%) para el Ensemble basados en el RMSE
rmse_ensemble = df_metricas.loc['Ensemble Adaptativo', 'RMSE']
z_score_95 = 1.96
predicciones_test['IC_Upper'] = predicciones_test['Ensemble'] + (z_score_95 * rmse_ensemble)
predicciones_test['IC_Lower'] = predicciones_test['Ensemble'] - (z_score_95 * rmse_ensemble)

# Visualización
plt.figure(figsize=(14, 7))

# Datos de Entrenamiento (Contexto)
plt.plot(train.index[-12:], train['Recaudo_Neto'].iloc[-12:], label='Datos Entrenamiento (Últimos 12m)', color='gray', linestyle='--')

# Datos Reales vs Predicción Ensemble
plt.plot(predicciones_test.index, predicciones_test['Real'], label='Recaudo Real (Test)', color='black', linewidth=2, marker='o')
plt.plot(predicciones_test.index, predicciones_test['Ensemble'], label='Predicción Ensemble', color='blue', linewidth=2.5)

# Intervalos de Confianza
plt.fill_between(predicciones_test.index, 
                 predicciones_test['IC_Lower'], 
                 predicciones_test['IC_Upper'], 
                 color='blue', alpha=0.15, label='Intervalo de Confianza 95%')

plt.title('Pronóstico de Rentas Cedidas: Ensemble Adaptativo (XGBoost + SARIMA + Prophet)', fontsize=16, fontweight='bold')
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Recaudo Neto ($)', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

# Sistema de Alerta de Eficiencia
print("--- SISTEMA DE ALERTA DE EFICIENCIA FISCAL ---")
alertas = predicciones_test[predicciones_test['Real'] < predicciones_test['IC_Lower']]

if len(alertas) > 0:
    print(f"⚠️ ALERTA ROJA DISPARADA: Se detectaron {len(alertas)} meses donde el recaudo real cayó por debajo del límite inferior esperado (IC 95%).")
    for fecha, row in alertas.iterrows():
        diferencia = row['IC_Lower'] - row['Real']
        print(f"  - {fecha.strftime('%Y-%m')}: Déficit de ${diferencia:,.0f} frente al límite inferior permitido.")
else:
    print("✅ SISTEMA SALUDABLE: El recaudo real se mantuvo dentro o por encima de los intervalos de confianza esperados. No hay alertas de ineficiencia en el periodo de prueba.")